In [1]:
from datasets import load_dataset

In [2]:
dataset = load_dataset('imdb')
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [3]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

In [4]:
from collections import Counter

counter = Counter()
for example in dataset['train']:
    counter.update(tokenize(example['text']))


In [5]:
VOCAB_SIZE = 10_000
vocab = {'PAD':0, 'UNK':1}
for word, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print(f'Dict len {len(vocab)}')

Dict len 10000


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

In [7]:
MAX_LEN = 256

class IMDBDataset(Dataset):
    def __init__(self, data, vocab, max_len):
        self.data = data
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data[idx]['text']
        label = self.data[idx]['label']

        tokens = tokenize(text)

        tokens = tokens[:self.max_len]
        ids = [self.vocab.get(t, 1) for t in tokens]
        padding = [0] * (self.max_len - len(ids))
        ids = ids + padding

        return torch.tensor(ids), torch.tensor(label)

In [8]:
train_dataset = IMDBDataset(dataset['train'], vocab, max_len=MAX_LEN)
test_dataset = IMDBDataset(dataset['test'], vocab, max_len=MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

tokens, label = train_dataset[0]
print(f'Tokens {len(tokens)}, Label {label}')

Tokens 256, Label 0


In [18]:
import torch.nn as nn

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=128, nhead=4, num_layers=4, dim_feedforward=256, num_classes=2, dropout=0.1):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=0,
        )

        self.position_embedding = nn.Embedding(
            max_len,
            d_model,
        )

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            self.encoder_layer,
            num_layers,
        )

        
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            d_model, 
            num_classes,
        )

    
    def forward(self, x):
        batch_size, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device = x.device
        ).unsqueeze(0).expand(batch_size, seq_len)

        x = (
            self.token_embedding(x)
            + self.position_embedding(positions)
        )

        x = self.transformer(x)
        x = x.mean(dim=1)
        x = self.dropout(x)
        return self.classifier(x)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = TransformerClassifier(vocab_size=VOCAB_SIZE, max_len=MAX_LEN).to(device)

print(model)

TransformerClassifier(
  (token_embedding): Embedding(10000, 128, padding_idx=0)
  (position_embedding): Embedding(256, 128)
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
    )
    (linear1): Linear(in_features=128, out_features=256, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=256, out_features=128, bias=True)
    (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear

In [19]:
from torchmetrics import Accuracy

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
acc = Accuracy(task='multiclass', num_classes=2).to(device)

In [20]:
from tqdm.std import tqdm

best_loss = float('inf')

patience = 3
counter = 0
epoch = 0

while True:
    epoch += 1
    print(f'Epoch: {epoch}\n--------------')

    model.train()
    train_loss = 0

    for X, y in tqdm(train_dataloader):
        X, y = X.to(device), y.to(device)
        y_pred = model(X)

        loss = loss_fn(y_pred, y)

        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss /= len(train_dataloader)

    model.eval()
    test_loss = 0

    with torch.inference_mode():
        for X, y in test_dataloader:
            X, y = X.to(device), y.to(device)

            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            acc.update(y, pred.argmax(dim=1))
            
        test_loss /= len(test_dataloader)
        test_acc = acc.compute()
        acc.reset()

    print(f'Train loss: {train_loss:.4f} | Test loss: {test_loss:.4f} | Accuracy: {test_acc * 100:.2f}%')

    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), 'best_imdb_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print('Early stopping')
            break

Epoch: 1
--------------


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:52<00:00,  7.41it/s]


Train loss: 0.6551 | Test loss: 0.5872 | Accuracy: 68.92%
Epoch: 2
--------------


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:53<00:00,  7.36it/s]


Train loss: 0.5444 | Test loss: 0.5200 | Accuracy: 74.07%
Epoch: 3
--------------


...

Train loss: 0.3067 | Test loss: 0.4071 | Accuracy: 81.82%
Epoch: 11
--------------


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:53<00:00,  7.35it/s]


Train loss: 0.2863 | Test loss: 0.4101 | Accuracy: 82.26%
Epoch: 12
--------------


100%|████████████████████████████████████████████████████████████████████████████████| 391/391 [00:52<00:00,  7.38it/s]


Train loss: 0.2709 | Test loss: 0.4222 | Accuracy: 82.50%
Early stopping
